In [1]:
import json
import typing
import ollama
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.runnables.graph import MermaidDrawMethod
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langgraph.graph import StateGraph, END
from pydantic import BaseModel 
from pprint import pprint
from IPython.display import Image, display

## Defining the open-source LLM with Ollama

In [2]:
llm_opensource = "llama3.2"

In [3]:
q = "Who are the top 10 most winners of the Premier League?"

q = "Who are the top 10 latest winners of Premier League?"

In [5]:
result = ollama.chat(model = llm_opensource, 
                        messages = [{"role":"system", "content":""},
                                    {"role":"user", "content":q}])

In [6]:
print(result.message.content)

Here are the top 10 latest winners of the Premier League, starting from the most recent season:

1. 2022-23 - Manchester City
2. 2021-22 - Manchester City
3. 2020-21 - Manchester City
4. 2019-20 - Liverpool
5. 2018-19 - Manchester City
6. 2017-18 - Manchester City
7. 2016-17 - Chelsea
8. 2015-16 - Leicester City
9. 2014-15 - Chelsea
10. 2013-14 - Manchester United

Note that the season duration and start date might vary from year to year, but these are the latest winners of the Premier League as of my cut-off knowledge in December 2023.


## Defining the Tools

In the LangGraph context, tools are external functions that the graph nodes can call to execute specific tasks, such as search in the web, interact with APIs and manipulate data. They allow the LangGraph conversation flow access external funcionalities, becoming the agent more dynamical and able to handle with different kind of requests.

In [7]:
@tool("search_tool")
def search_tool(q:str) -> str:
    """Search on DuckDuckGo navigator using `q` as input"""
    return DuckDuckGoSearchRun().run(q)

In [8]:
# Testing
print(search_tool.invoke(q))

2 days ago -Only five clubs have won the title in three consecutive seasons:Huddersfield Town, Arsenal, Liverpool, Manchester United (twice) and Manchester City, with the latter being the only club to have won four successive titles. ... 24 clubs which have won the English top level title, including 7 ... 1 day ago -Twenty clubs are competing in the 2025–26 season – top seventeen from the previous season and three promoted from the Championship. Leicester City, Ipswich Town and Southampton were relegated to the EFL Championship for the 2025–26 season, whilstLeeds United, Burnley, and Sunderland, as ... 1 week ago -After Arsenal led the division for much of the 2007–08 season,Manchester Unitedretained the championship on the final day of the season and won their eleventh Premier League title in 2008–09 after much competition from Liverpool. Chelsea reclaimed the league in 2009–10, scoring a record ... 1 week ago -Premier League · 20 20 13 10 9 7 6 6 4 4 3 3 3 3 2 2 2 2 2 1 1 1 1 1 · Tou

In [9]:
@tool("final_answer_tool")
def final_answer_tool(text:str) -> str:
    """
    Return a natural language answer to user using the `input text`.
    You must provide the maximum of possible context and specify the source of the information.
    """

    return text

In [10]:
# Testing
print(final_answer_tool.invoke("Test") )

Test


In [11]:
# Tools dict
dic_tools = {"search_tool":search_tool, 
             "final_answer_tool":final_answer_tool}

## Adding tools to help the decision-making processo of the Agent

In [12]:
prompt = """
You are a specialist in news, you must answer all the questions of the user, you can use the tools list provided to you.
Your goal is provide the best possible answer to the user, including important informations about the sources and tools used.

Look, when you call a tool, you need to provide the name of the tool and the argument that it uses in JSON format.
For each call, you MUST use ONLY on tool and the answer format ALWAYS NEEDS to follow this patter:
```json
{"name": "<tool_name>", "parameters": "<tool_input_key>":"<tool_input_value>"}
```

You must remember, You MUST NOT use the tool with the same query more than once.
You must remember, if the user doesn't make a specific question, you MUST use the tool `final_answer_tool` directly.

Everytime the user ask you a question, you need to store in your memory.
Everytime you find some information related to user question, you have to store in your memory.

You must try collect informations from several sources before providing a answer to the user.
After collect enough informations to answer the user question, use the tool `final_answer_tool`.
"""

In [13]:
str_tools = "\n".join([str(n+1)+". `"+str(v.name)+"`: "+str(v.description) for n,v in enumerate(dic_tools.values())])

In [14]:
print(str_tools)

1. `search_tool`: Search on DuckDuckGo navigator using `q` as input
2. `final_answer_tool`: Return a natural language answer to user using the `input text`.
You must provide the maximum of possible context and specify the source of the information.


In [15]:
# Tools prompt
prompt_tools = f"You can use the following tools:\n{str_tools}"

print(prompt_tools)

You can use the following tools:
1. `search_tool`: Search on DuckDuckGo navigator using `q` as input
2. `final_answer_tool`: Return a natural language answer to user using the `input text`.
You must provide the maximum of possible context and specify the source of the information.


In [16]:
llm_answer = ollama.chat(model = llm_opensource, messages = [{"role":"system", "content":prompt+"\n"+prompt_tools}, {"role":"user", "content":q}], format = "json")

llm_answer["message"]["content"]

'{"name": "search_tool", "parameters": {"q": "{\\"PremierLeague\\": \\"top 10 latest winners\\"}"}}'

In [17]:
content = llm_answer["message"]["content"]
data = json.loads(content)

parameters = data.get("parameters", {})
print("Keys in 'parameters':", parameters.keys())

# try to get 'q' value or the input text
tool_input = parameters.get('q') or parameters.get('input text')
if tool_input is None:
    raise KeyError("No one of the keys 'q' or 'input text' were found in 'parameter'.")

print("Value of the tool_input:", tool_input)

Keys in 'parameters': dict_keys(['q'])
Value of the tool_input: {"PremierLeague": "top 10 latest winners"}


In [18]:
# Testing the tool with the context
context = search_tool.invoke(tool_input)
print("Context:\n", context)

Context:
 2 days ago -Only five clubs have won the title in three consecutive seasons:Huddersfield Town, Arsenal, Liverpool, Manchester United (twice) and Manchester City, with the latter being the only club to have won four successive titles. ... 24 clubs which have won the English top level title, including 7 ... 1 day ago -Italics indicate former Premier ... the Premier League. Twenty clubs are competing in the 2025–26 season – top seventeen from the previous season and three promoted from the Championship. Leicester City, Ipswich Town and Southampton were relegated to the EFL Championship for the 2025–26 season, whilst Leeds United, Burnley, and Sunderland, as winners, runners-up ... 1 week ago -Managers who won the Premier League in their debut season: 5, Jose Mourinho for Chelsea F.C. in 2004–05, Carlo Ancelotti for Chelsea F.C. in 2009–10, Manuel Pellegrini for Manchester City in 2013–14, Antonio Conte for Chelsea F.C. in 2016–17, Arne Slot for Liverpool F.C. in 2024–25) ^ Becau

## Creating the Agent Structure
On LangGraph, an agent can be developed as a class. This is useful to encapsulate the agent logic, including the nodes definition, trasitions and the memory use and tool.

When defining an agent as a class, you can:
- Structure better the source, spliting the logic and graph execution.
- Storing internal states and configure the decision flow in a more organized way.
- Facilitate the reuse and the scalability of the agent.

In [19]:
# AgentClass
class MyAgent(BaseModel):
    tool_name: str # Tool name used by the agent
    tool_input: dict # Input provided by the agent (parameters)
    tool_output: str | None = None  # Output result
    
    # Method to crate the instance from the model answer (LLM)
    @classmethod
    def from_llm(cls, res: dict):  
        try:
            out = json.loads(res["message"]["content"])
            return cls(tool_name=out["name"], tool_input=out["parameters"])
        except Exception as e:
            print(f"Ollama error:\n{res}\n")
            raise e

In [20]:
# Testing the agent
my_agent = MyAgent.from_llm(llm_answer)
print("Initial format:\n", llm_answer["message"]["content"], "\nAgent format:")
my_agent

Initial format:
 {"name": "search_tool", "parameters": {"q": "{\"PremierLeague\": \"top 10 latest winners\"}"}} 
Agent format:


MyAgent(tool_name='search_tool', tool_input={'q': '{"PremierLeague": "top 10 latest winners"}'}, tool_output=None)

In [21]:
MyAgent(tool_name = "dsa_toosearch_tooll_busca", 
         tool_input = {'q':'Who are the top 10 latest winners of Premier League?'}, 
         tool_output = str( search_tool.invoke(tool_input)) )

MyAgent(tool_name='dsa_toosearch_tooll_busca', tool_input={'q': 'Who are the top 10 latest winners of Premier League?'}, tool_output='2 days ago -Only five clubs have won the title in three consecutive seasons:Huddersfield Town, Arsenal, Liverpool, Manchester United (twice) and Manchester City, with the latter being the only club to have won four successive titles. ... 24 clubs which have won the English top level title, including 7 ... 1 day ago -Italics indicate former Premier ... the Premier League. Twenty clubs are competing in the 2025–26 season – top seventeen from the previous season and three promoted from the Championship. Leicester City, Ipswich Town and Southampton were relegated to the EFL Championship for the 2025–26 season, whilst Leeds United, Burnley, and Sunderland, as winners, runners-up ... 1 week ago -Managers who won the Premier League in their debut season: 5, Jose Mourinho for Chelsea F.C. in 2004–05, Carlo Ancelotti for Chelsea F.C. in 2009–10, Manuel Pellegrini

## Setting the System Memory

On LangGraph, the memory refers the ability to store and retrieve informations through the conversation flow. This allows that the agent keep the context among the interactions, and adjust its answers based on previous interactions. The memory can be implemented in different ways, such as storing variables on the graph state or integrating external storage solutions.

In [22]:
# Function to generate contextual memory in th Agent 
def agent_memory(lst_res: list[MyAgent], user_q: str) -> list:
    memory = []
    
    # Iterate about the previous answers that having a valid output
    for res in [res for res in lst_res if res.tool_output is not None]:
        
        memory.extend([
            {"role": "assistant", "content": json.dumps({"name": res.tool_name, "parameters": res.tool_input})},
            {"role": "user", "content": res.tool_output}
        ])
    
    # If the memory is not empty, add a reminder of the original answer 
    if memory:
        memory += [{"role": "user", "content": (f'''
                This is only a reminder that my original query was `{user_q}`.
                Responda apenas à pergunta original e nada mais, mas use as informações que lhe dei.
                Answer only the original question and nothing else, but use the informations I gave you.
                Provide the maximum of possible informations when you use the tool `final_answer`.
                ''')}]

    return memory

In [23]:
chat_history = [{"role": "user", "content": "Hi, how are you doing?"},
                {"role": "assistant", "content": "I'm great, thanks!"},
                {"role": "user", "content": "I have a question"},
                {"role": "assistant", "content": "Yes, what would you like to know?"}]

In [24]:
def execute_agent(prompt: str, 
                       dic_tools: dict, 
                       user_q: str, 
                       chat_history: list[dict], 
                       lst_res: list[MyAgent]) -> MyAgent:
    
    # Initializing the memory with the previous answers from the agent
    memory = agent_memory(lst_res = lst_res, user_q = user_q)
    
    if memory:
        tools_used = [res.tool_name for res in lst_res]
        if len(tools_used) >= len(dic_tools.keys()):
            memory[-1]["content"] = "Now you must to use the tool `final_answer_tool`."

    str_tools = "\n".join([
        f"{n+1}. `{v.name}`: {v.description}" 
        for n, v in enumerate(dic_tools.values())
    ])
    
    prompt_tools = f"You can use the following tools:\n{str_tools}"
        
    # Build the full messages list (prompt + history + memory)
    messages = [
        {"role": "system", "content": prompt + "\n" + prompt_tools},
        *chat_history,
        {"role": "user", "content": user_q},
        *memory
    ]
    
    llm_res = ollama.chat(model = llm_opensource, 
                          messages = messages, 
                          format = "json")
    
    return MyAgent.from_llm(llm_res)

In [25]:
# Testng
agent_res = execute_agent(prompt = prompt, 
                               dic_tools = dic_tools, 
                               user_q = q, 
                               chat_history = chat_history, 
                               lst_res = [])
print("Agent answer:", agent_res)

Agent answer: tool_name='search_tool' tool_input={'q': '{" Premiershipl Latest Winner":'} tool_output=None


## Creating the Graph Workflow Components

The LangGraph graph workflow is a structure that defines how the interactions flow between the graph nodes. It represents the agent logic as in a directed graph, where each node can execute a function and each edge defines possibles transictions to the next steps. 

In [26]:
# State Class
class State(typing.TypedDict):
    user_q: str
    chat_history: list 
    lst_res: list[MyAgent] # Previous answers

    output: dict

In [27]:
# Testando
state = State({"user_q":q, "chat_history":chat_history, "lst_res":[agent_res], "output":{}})
state

{'user_q': 'Who are the top 10 latest winners of Premier League?',
 'chat_history': [{'role': 'user', 'content': 'Hi, how are you doing?'},
  {'role': 'assistant', 'content': "I'm great, thanks!"},
  {'role': 'user', 'content': 'I have a question'},
  {'role': 'assistant', 'content': 'Yes, what would you like to know?'}],
 'lst_res': [MyAgent(tool_name='search_tool', tool_input={'q': '{" Premiershipl Latest Winner":'}, tool_output=None)],
 'output': {}}

## Node
On LangGraph, a node is a processing unit inside the Graph Workflow. Each node executes a specific function and can modify the state before passing the execution to another node.

In [28]:
def agent_node(state):
    
    print("--- Agent Node ---")
    
    agent_res = execute_agent(
        prompt = prompt, 
        dic_tools = {k: v for k, v in dic_tools.items() if k in ["search_tool", "final_answer_tool"]},
        user_q = state["user_q"], 
        chat_history = state["chat_history"], 
        lst_res = state["lst_res"]
    )
    
    print(agent_res)
    
    return {"lst_res": [agent_res]}

In [29]:
# Testing
agent_node(state)

--- Agent Node ---
tool_name='search_tool' tool_input={'q': 'latest premier league winners'} tool_output=None


{'lst_res': [MyAgent(tool_name='search_tool', tool_input={'q': 'latest premier league winners'}, tool_output=None)]}

### Specialized Node

The node tool agent_node is a specialized node on LangGraph responsible to process tools calls inside the agent flow. He works as a intermediate that:
* Identify which tool was called based on the state `(state["lst_res"][-1])`;
* Extracts and validates the necessary parameters to the tool work correctly;
* Invoke the right tool `(dic_tools[res.tool_name].invoke(tool_input))`;
* Return the right output according to the used tool.

In [30]:
def agent_node_tool(state):
    
    print("--- Agent Node Tool ---")
    res = state["lst_res"][-1]
    
    print(f"{res.tool_name}(input = {res.tool_input})")

    if res.tool_name == "final_answer_tool":
        possible_keys = ['text', 'input text', 'q']
        tool_input = None
        for key in possible_keys:
            if key in res.tool_input:
                tool_input = res.tool_input[key]
                break
        
        if not tool_input:
            raise ValueError(f"Necessary paramater to '{res.tool_name} was not found'. Expected keys: {possible_keys}. Current keys: {list(res.tool_input.keys())}")
        
        agent_res = MyAgent(
            tool_name = res.tool_name,
            tool_input = {'text': tool_input},
            tool_output = str(dic_tools[res.tool_name].invoke(tool_input))
        )
    else:
        possible_keys = ['q', 'input text', 'text']
        tool_input = None
        for key in possible_keys:
            if key in res.tool_input:
                tool_input = res.tool_input[key]
                break
        
        if not tool_input:
            raise ValueError(f"Necessary paramater to '{res.tool_name} was not found'. Expected keys: {possible_keys}. Current keys: {list(res.tool_input.keys())}")
        
        # Cria a resposta do agente com outras ferramentas (não final_answer_tool)
        agent_res = MyAgent(
            tool_name = res.tool_name,
            tool_input = {'q': tool_input},
            tool_output = str(dic_tools[res.tool_name].invoke(tool_input))
        )

    # Returns the final output if it is 'final_answer_tool', if not, returns to continue de process
    return {"output": agent_res} if res.tool_name == "final_answer_tool" else {"lst_res": [agent_res]}

In [31]:
# Testing
agent_node_tool(state)

--- Agent Node Tool ---
search_tool(input = {'q': '{" Premiershipl Latest Winner":'})


{'lst_res': [MyAgent(tool_name='search_tool', tool_input={'q': '{" Premiershipl Latest Winner":'}, tool_output='2 days ago -The English football champions are the annual winners of the top-tier competition in the English football league system. Following the codification of professional football by the Football Association in 1885, the Football League was established in 1888, after meetings initiated by Aston Villa ... 1 day ago -Leicester City, Ipswich Town and Southampton were relegated to the EFL Championship for the 2025–26 season, whilstLeeds United, Burnley, and Sunderland, as winners, runners-up and play-off final winners, respectively, were promoted from the ... 2 weeks ago -The 2024–25 Premier League was the 33rd season of the Premier League and the 126th season of top-flight English football overall.Manchester Cityentered the season as four-time defending champions, but were dethroned by Liverpool, who emerged as Premier League winners with four games to ... 1 week ago -This 

### Conditional Edges
The conditional edges determine dynamically which node will be the next to be exectuded in the flow, based on the current state. They allow the graph follow different paths depending the definitive logic. 

In [32]:
def condicional_edges(state):
    print("--- Edges Condicionais ---")
    
    last_res = state["lst_res"][-1]
    # Decides which node will be the next based the used tool 
    next_node = last_res.tool_name if isinstance(state["lst_res"], list) else "dsa_tool_resposta_final"
    
    print("Next Node:", next_node)    
    return next_node

In [34]:
# Testing
condicional_edges(state)

--- Edges Condicionais ---
Next Node: search_tool


'search_tool'